# Day 4 — Preparing Data Sets I: Importing from Multiple Sources
Prepare data sets (import data)

Run the **Setup** cell below first — it generates every file you need for today
(`service_requests.csv` + 3 broken variants, and `regional_offices.json`) right
inside this Colab session. Nothing to upload.


In [1]:
# ============================================================
# SETUP — run this first. It generates every data file used
# today (service_requests.csv + 3 broken variants, and
# regional_offices.json) directly inside this Colab session,
# so there is nothing to upload.
# ============================================================
import json, csv, os, random
import pandas as pd

random.seed(42)
os.makedirs("data", exist_ok=True)

regions = [
    {"region_code": "NCR", "region_name": "National Capital Region", "offices": [
        {"office_id": "OFF-001", "office_name": "Manila Branch",
         "contact": {"phone": "8-527-1001", "email": "manila.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Civil Registry", "Tax Clearance"]},
        {"office_id": "OFF-002", "office_name": "Quezon City Branch",
         "contact": {"phone": "8-988-1002", "email": "qc.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Zoning Clearance"]},
        {"office_id": "OFF-003", "office_name": "Makati Branch",
         "contact": {"phone": "8-892-1003", "email": "makati.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Civil Registry"]},
    ]},
    {"region_code": "R3", "region_name": "Central Luzon", "offices": [
        {"office_id": "OFF-101", "office_name": "San Fernando Branch",
         "contact": {"phone": "45-961-1101", "email": "sanfernando.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Agricultural Permit"]},
        {"office_id": "OFF-102", "office_name": "Angeles Branch",
         "contact": {"phone": "45-322-1102", "email": "angeles.branch@example.gov.ph"},
         "services_offered": ["Civil Registry", "Tax Clearance"]},
    ]},
    {"region_code": "R4A", "region_name": "CALABARZON", "offices": [
        {"office_id": "OFF-201", "office_name": "Calamba Branch",
         "contact": {"phone": "49-545-1201", "email": "calamba.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Zoning Clearance", "Tax Clearance"]},
        {"office_id": "OFF-202", "office_name": "Batangas City Branch",
         "contact": {"phone": "43-723-1202", "email": "batangas.branch@example.gov.ph"},
         "services_offered": ["Civil Registry"]},
        {"office_id": "OFF-203", "office_name": "Antipolo Branch",
         "contact": {"phone": "2-697-1203", "email": "antipolo.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Agricultural Permit"]},
    ]},
    {"region_code": "R7", "region_name": "Central Visayas", "offices": [
        {"office_id": "OFF-301", "office_name": "Cebu City Branch",
         "contact": {"phone": "32-255-1301", "email": "cebu.branch@example.gov.ph"},
         "services_offered": ["Business Permit", "Civil Registry", "Zoning Clearance"]},
        {"office_id": "OFF-302", "office_name": "Mandaue Branch",
         "contact": {"phone": "32-346-1302", "email": "mandaue.branch@example.gov.ph"},
         "services_offered": ["Tax Clearance"]},
    ]},
]

with open("data/regional_offices.json", "w", encoding="utf-8") as f:
    json.dump({"regions": regions}, f, indent=2, ensure_ascii=False)

all_office_ids = [o["office_id"] for r in regions for o in r["offices"]]
categories = ["Business Permit", "Civil Registry", "Tax Clearance", "Zoning Clearance", "Agricultural Permit", "Complaint"]
statuses = ["Open", "In Progress", "Resolved", "Closed"]
priorities = ["Low", "Medium", "High"]
descriptions = [
    "Requesting renewal of annual permit", "Application for new registration",
    "Follow-up on pending clearance", "Cafe owner requesting inspection schedule",
    "Resident reporting drainage issue near the plaza", "Request for certified true copy of records",
    "Inquiry on requirements for a new application", "Complaint regarding delayed processing",
    "Request to update contact information on file", "Application for special agricultural permit",
]
dates = [f"2026-0{m}-{d:02d}" for m in (7, 8) for d in (3, 8, 12, 15, 19, 22, 27)]

rows, n = [], 1
for i in range(24):
    rows.append({"request_id": f"REQ-{n:04d}", "date_submitted": dates[i % len(dates)],
                 "office_id": all_office_ids[i % len(all_office_ids)], "category": categories[i % len(categories)],
                 "description": descriptions[i % len(descriptions)], "status": statuses[i % len(statuses)],
                 "priority": priorities[i % len(priorities)]})
    n += 1
rows.append({"request_id": f"REQ-{n:04d}", "date_submitted": "2026-08-05", "office_id": "OFF-999",
             "category": "Business Permit", "description": "Old request routed to a since-closed satellite office",
             "status": "Open", "priority": "Medium"}); n += 1
rows.append({"request_id": f"REQ-{n:04d}", "date_submitted": "2026-08-11", "office_id": "OFF-02",
             "category": "Zoning Clearance", "description": "Zoning clearance request with office code entered by staff",
             "status": "In Progress", "priority": "Low"}); n += 1
rows.append({"request_id": f"REQ-{n:04d}", "date_submitted": "2026-08-14", "office_id": "OFF-999",
             "category": "Civil Registry", "description": "Second request affected by the same decommissioned office code",
             "status": "Open", "priority": "High"})

fieldnames = ["request_id", "date_submitted", "office_id", "category", "description", "status", "priority"]

def write_csv(path, data_rows, delimiter=",", encoding="utf-8", extra_first_line=None):
    with open(path, "w", newline="", encoding=encoding) as f:
        if extra_first_line:
            f.write(extra_first_line + "\n")
        w = csv.DictWriter(f, fieldnames=fieldnames, delimiter=delimiter)
        w.writeheader()
        for r in data_rows:
            w.writerow(r)

write_csv("data/service_requests.csv", rows)
write_csv("data/service_requests_semicolon.csv", rows, delimiter=";")

rows_latin1 = [dict(r) for r in rows]
rows_latin1[3]["description"] = "Café owner requesting inspection schedule for Peñalosa St."
rows_latin1[8]["description"] = "Request to update contact information - Ñoñoy Cruz, applicant"
write_csv("data/service_requests_latin1.csv", rows_latin1, encoding="latin-1")

write_csv("data/service_requests_extrarow.csv", rows, extra_first_line="Q3 2026 Service Requests Export - Internal Use Only")

# --- A third source: a SQL table (this is the "previous topic" callback) ---
# category_sla lives in a SQLite database, the way a lot of real reference
# data lives in an actual database rather than a flat file. Deliberately
# does NOT include a row for "Complaint" -- complaints are logged in a
# separate system with no formal SLA, so that merge will have a real
# unmatched-key case too.
import sqlite3

conn = sqlite3.connect("data/workshop.db")
conn.execute("DROP TABLE IF EXISTS category_sla")
conn.execute("""
    CREATE TABLE category_sla (
        category TEXT PRIMARY KEY,
        department TEXT,
        sla_days INTEGER,
        priority_weight INTEGER
    )
""")
category_sla_rows = [
    ("Business Permit", "Licensing", 5, 2),
    ("Civil Registry", "Civil Registry Office", 3, 1),
    ("Tax Clearance", "Treasury", 4, 2),
    ("Zoning Clearance", "Planning", 7, 3),
    ("Agricultural Permit", "Agriculture", 10, 3),
    # "Complaint" intentionally omitted
]
conn.executemany("INSERT INTO category_sla VALUES (?, ?, ?, ?)", category_sla_rows)
conn.commit()
conn.close()

print("Data ready in ./data/:")
for fn in sorted(os.listdir("data")):
    print(" -", fn)


def check(value, condition_fn, success_msg, hint_msg):
    """Self-check helper used throughout this notebook.
    Run the cell above, then run the check cell below it:
      - "not attempted yet"  -> you still need to replace the None
      - ✅ green check         -> correct, move on
      - ❌ red X                -> re-read the hint and try again
    """
    if value is None:
        print("Not attempted yet - replace the None above with your code, then re-run this cell.")
        return
    try:
        ok = condition_fn(value)
    except Exception as e:
        print(f"Your code ran, but checking it raised an error: {e}")
        return
    if ok:
        print("Correct -", success_msg)
    else:
        print("Not quite -", hint_msg)


Data ready in ./data/:
 - regional_offices.json
 - service_requests.csv
 - service_requests_extrarow.csv
 - service_requests_latin1.csv
 - service_requests_semicolon.csv
 - workshop.db


In [2]:
import pandas as pd
import json

## Lecture 1 — Data Sources and Formats

CSV structure, delimiters, encoding, and headers — and what can go wrong on import.

**The "first five minutes" checklist** (use this after every import):
```python
df.shape            # row/column count sanity check
df.head()           # does it look like the source data?
df.dtypes           # are numeric/date columns typed correctly?
df.isna().sum()     # unexpected nulls?
df.columns.tolist() # stray, duplicate, or padded column names?
```


## Hands-On 1 — Import, Break, Diagnose

Work through this section **top to bottom, one cell at a time**. Every TODO
cell is followed by a "check" cell — run it right after to see whether you
got it right. It's fine to fail a check a few times; that's the exercise.

**Required variable names** (the checks look for these exact names):
`df`, `semicolon_broken`, `semicolon_fixed`, `latin1_fixed`,
`extrarow_broken`, `extrarow_fixed`


In [3]:
# Step 1 — Baseline import
# This one's done for you — it's the "known good" file everything else
# gets compared against.
df = pd.read_csv("data/service_requests.csv")

# Now YOU run the "first five minutes" checklist from Lecture 1 on df.
# Write one line per check (there should be 5 lines total):
#   1. shape
#   2. head()
#   3. dtypes
#   4. isna().sum()
#   5. columns.tolist()

# 1. Check the number of rows and columns
df.shape

# 2. Preview the first 5 rows
df.head()

# 3. Check the data types of each column
df.dtypes

# 4. Check the number of missing values in each column
df.isna().sum()

# 5. List all column names
df.columns.tolist()

['request_id',
 'date_submitted',
 'office_id',
 'category',
 'description',
 'status',
 'priority']

**Step 2 — semicolon delimiter.**

In [10]:
# Step 2a — Break it on purpose: semicolon-delimited file
#
# TODO: import data/service_requests_semicolon.csv using plain pd.read_csv()
#       -- no extra parameters yet -- and assign the result to semicolon_broken.

# Intentionally use the default comma delimiter.
semicolon_broken = pd.read_csv("data/service_requests_semicolon.csv")

print(semicolon_broken.shape if semicolon_broken is not None else "not attempted yet")


(27, 1)


In [11]:
check(
    semicolon_broken,
    lambda d: d.shape[1] == 1,
    "everything landed in one column — that's your delimiter clue.",
    "expected exactly 1 column when the wrong delimiter is assumed. "
    "Did you call pd.read_csv() with no extra arguments?"
)


Correct - everything landed in one column — that's your delimiter clue.


In [14]:
# Step 2b — Diagnose, then fix
#
# Look at semicolon_broken.columns.tolist() -- what character do you see
# still sitting inside that single column name? That's your delimiter.
#
# TODO: re-import the same file, this time passing the correct delimiter
#       to the `sep=` parameter. Assign the result to semicolon_fixed.

# The file uses ";" instead of the default comma "," as its delimiter.
semicolon_fixed = pd.read_csv(
    "data/service_requests_semicolon.csv",
    sep=";"
)

semicolon_fixed


,request_id,date_submitted,office_id,category,description,status,priority
0,REQ-0001,2026-07-03,OFF-001,Business Permit,Requesting renewal of annual permit,Open,Low
1,REQ-0002,2026-07-08,OFF-002,Civil Registry,Application for new registration,In Progress,Medium
2,REQ-0003,2026-07-12,OFF-003,Tax Clearance,Follow-up on pending clearance,Resolved,High
3,REQ-0004,2026-07-15,OFF-101,Zoning Clearance,Cafe owner requesting inspection schedule,Closed,Low
4,REQ-0005,2026-07-19,OFF-102,Agricultural Permit,Resident reporting drainage issue near the plaza,Open,Medium
5,REQ-0006,2026-07-22,OFF-201,Complaint,Request for certified true copy of records,In Progress,High
6,REQ-0007,2026-07-27,OFF-202,Business Permit,Inquiry on requirements for a new application,Resolved,Low
7,REQ-0008,2026-08-03,OFF-203,Civil Registry,Complaint regarding delayed processing,Closed,Medium
8,REQ-0009,2026-08-08,OFF-301,Tax Clearance,Request to update contact information on file,Open,High
9,REQ-0010,2026-08-12,OFF-302,Zoning Clearance,Application for special agricultural permit,In Progress,Low


In [16]:
check(
    semicolon_fixed,
    lambda d: d.shape == df.shape and list(d.columns) == list(df.columns),
    "shape and columns now match the baseline -- fixed!",
    f"not matching yet -- baseline is {df.shape} with columns {list(df.columns)}. "
    "Double-check the character you passed to sep=."
)


Correct - shape and columns now match the baseline -- fixed!


**Step 3 — wrong encoding.**

In [18]:
# Step 3a — Break it on purpose: wrong encoding
#
# TODO: uncomment the line below and run this cell. It is SUPPOSED to
#       crash -- read the error message pandas gives you before moving on.
#       (What kind of error is it? What does it say it couldn't decode?)

# broken_latin1 = pd.read_csv("data/service_requests_latin1.csv")

# Error: UnicodeDecodeError: 'utf-8' codec can't decode byte ...

In [20]:
# Step 3b — Fix it
#
# TODO: re-import data/service_requests_latin1.csv, this time passing the
#       correct value to `encoding=` (Lecture 1 named two common options
#       for exactly this situation). Assign the result to latin1_fixed.


# The file was saved using Latin-1 encoding.
latin1_fixed = pd.read_csv(
    "data/service_requests_latin1.csv",
    encoding="latin-1"
)

# Check the shape
print(latin1_fixed.shape)

(27, 7)


In [21]:
check(
    latin1_fixed,
    lambda d: d.shape == df.shape,
    "loaded successfully and matches the baseline shape!",
    f"not matching yet -- baseline is {df.shape}. Try a different encoding= value."
)


Correct - loaded successfully and matches the baseline shape!


**Step 4 — extra title row.**

In [22]:
# Step 4a — Break it on purpose: extra title row
#
# TODO: import data/service_requests_extrarow.csv with plain pd.read_csv()
#       (no extra parameters yet) and assign it to extrarow_broken.

# Step 4a — Break it on purpose: extra title row
# Intentionally use the default import settings.
extrarow_broken = pd.read_csv("data/service_requests_extrarow.csv")

print(extrarow_broken.shape if extrarow_broken is not None else "not attempted yet")
print(extrarow_broken.columns.tolist() if extrarow_broken is not None else "")

(28, 1)
['Q3 2026 Service Requests Export - Internal Use Only']


In [23]:
check(
    extrarow_broken,
    lambda d: d.shape[1] == 1,
    "the title row got treated as the header, so everything else collapsed into one column.",
    "expected exactly 1 column here too -- for a different reason than Step 2. "
    "Print extrarow_broken.columns.tolist() and see what that single column is named."
)


Correct - the title row got treated as the header, so everything else collapsed into one column.


In [24]:
# Step 4b — Fix it
#
# TODO: re-import the file, this time telling pandas to skip the first
#       line before it looks for the header row. Assign to extrarow_fixed.

# Skip the first line so pandas uses the actual CSV header.
extrarow_fixed = pd.read_csv(
    "data/service_requests_extrarow.csv",
    skiprows=1
)

# Check the result
print(extrarow_fixed.shape)
print(extrarow_fixed.columns.tolist())


(27, 7)
['request_id', 'date_submitted', 'office_id', 'category', 'description', 'status', 'priority']


In [25]:
check(
    extrarow_fixed,
    lambda d: d.shape == df.shape and list(d.columns) == list(df.columns),
    "shape and columns now match the baseline -- fixed!",
    f"not matching yet -- baseline is {df.shape} with columns {list(df.columns)}. "
    "How many lines sit above the real header in this file?"
)


Correct - shape and columns now match the baseline -- fixed!


**Step 5 — confirm everything together.**

In [26]:
# Step 5 — Confirm all three fixes at once
# Nothing to fill in here -- if Steps 2, 3, and 4 are correct, this should
# print True for all three.

for name, fixed in [
    ("semicolon", semicolon_fixed),
    ("latin1", latin1_fixed),
    ("extrarow", extrarow_fixed),
]:
    if fixed is None:
        print(f"{name}: not attempted yet")
    else:
        print(f"{name}: matches baseline shape = {fixed.shape == df.shape}")


semicolon: matches baseline shape = True
latin1: matches baseline shape = True
extrarow: matches baseline shape = True


## Lecture 2 — Working with JSON

Nested objects and arrays, how they differ from flat tables, and flattening
nested structures into a DataFrame with `pd.json_normalize()`.

Ask three questions before flattening any nested JSON:
1. What is the **record** I want one row per?
2. What is the **path** to the array containing those records? (`record_path`)
3. What outer fields should carry onto every row? (`meta`)


## Hands-On 2 — Load and Flatten Regional Offices

Same rhythm as Hands-On 1: fill in a TODO cell, then run the check cell
right after it.

**Required variable names:** `offices_data`, `first_office_email`, `offices`


In [31]:
# Step 1 — Load and inspect
# This part's given: it reads the raw JSON into a Python dict.
with open("data/regional_offices.json") as f:
    offices_data = json.load(f)

# TODO: print the first ~800 characters of the file, nicely indented, so
#       you can see the nesting. (Hint: json.dumps(..., indent=2))

print(json.dumps(offices_data, indent=2)[:800])

{
  "regions": [
    {
      "region_code": "NCR",
      "region_name": "National Capital Region",
      "offices": [
        {
          "office_id": "OFF-001",
          "office_name": "Manila Branch",
          "contact": {
            "phone": "8-527-1001",
            "email": "manila.branch@example.gov.ph"
          },
          "services_offered": [
            "Business Permit",
            "Civil Registry",
            "Tax Clearance"
          ]
        },
        {
          "office_id": "OFF-002",
          "office_name": "Quezon City Branch",
          "contact": {
            "phone": "8-988-1002",
            "email": "qc.branch@example.gov.ph"
          },
          "services_offered": [
            "Business Permit",
            "Zoning Clearance"
          ]
        },
  


In [32]:
check(
    offices_data,
    lambda d: "regions" in d and isinstance(d["regions"], list),
    "loaded correctly -- offices_data is a dict with a top-level 'regions' list.",
    "offices_data should be a dict containing a 'regions' key. Did Step 1's read succeed?"
)


Correct - loaded correctly -- offices_data is a dict with a top-level 'regions' list.


In [33]:
# Step 2 — Navigate manually before automating
#
# Before letting json_normalize do the work, prove to yourself you can
# find one value by hand. offices_data["regions"] is a list of regions;
# each region has an "offices" list; each office has a "contact" dict.
#
# TODO: get the email address of the FIRST office in the FIRST region.

first_office_email = offices_data["regions"][0]["offices"][0]["contact"]["email"]

print(first_office_email)


manila.branch@example.gov.ph


In [34]:
check(
    first_office_email,
    lambda v: v == offices_data["regions"][0]["offices"][0]["contact"]["email"],
    "that's the right email -- you navigated the nesting correctly.",
    "not quite the right value yet. Remember: region -> offices[0] -> contact -> email."
)


Correct - that's the right email -- you navigated the nesting correctly.


In [35]:
# Step 3 — Flatten with pd.json_normalize()
#
# You need to tell json_normalize three things (this is the "three
# questions" from Lecture 2):
#   1. What are the records?        -> pass offices_data["regions"] as the data
#   2. Where's the array to explode? -> record_path=...
#   3. What outer fields carry onto every row? -> meta=[...]
#
# TODO: fill in record_path and meta below.

offices = pd.json_normalize(
    offices_data["regions"],
    record_path="offices",
    meta=["region_code", "region_name"],
    sep="_",
)

print(offices)

  office_id           office_name  \
0   OFF-001         Manila Branch   
1   OFF-002    Quezon City Branch   
2   OFF-003         Makati Branch   
3   OFF-101   San Fernando Branch   
4   OFF-102        Angeles Branch   
5   OFF-201        Calamba Branch   
6   OFF-202  Batangas City Branch   
7   OFF-203       Antipolo Branch   
8   OFF-301      Cebu City Branch   
9   OFF-302        Mandaue Branch   

                                    services_offered contact_phone  \
0   [Business Permit, Civil Registry, Tax Clearance]    8-527-1001   
1                [Business Permit, Zoning Clearance]    8-988-1002   
2                  [Business Permit, Civil Registry]    8-892-1003   
3             [Business Permit, Agricultural Permit]   45-961-1101   
4                    [Civil Registry, Tax Clearance]   45-322-1102   
5  [Business Permit, Zoning Clearance, Tax Cleara...   49-545-1201   
6                                   [Civil Registry]   43-723-1202   
7             [Business Permit, 

In [36]:
check(
    offices,
    lambda d: d.shape[0] == sum(len(r["offices"]) for r in offices_data["regions"])
              and "region_code" in d.columns and "office_id" in d.columns,
    "flattened correctly -- one row per office, with region fields carried onto each row.",
    "not quite -- check record_path (should point at the offices list) and "
    "meta (should be the two outer region_ fields you want on every row)."
)


Correct - flattened correctly -- one row per office, with region fields carried onto each row.


**Step 4 — validate.**

In [37]:
# Step 4 — Validate
# TODO: run the same "first five minutes" checklist on `offices` that you
#       used in Hands-On 1 (shape, head, dtypes, isna().sum(), columns).
#       Does the row count match the number of offices you counted in the JSON?

# Step 4 — Validate

print("Shape:")
print(offices.shape)

print("\nFirst 5 rows:")
print(offices.head())

print("\nData types:")
print(offices.dtypes)

print("\nMissing values:")
print(offices.isna().sum())

print("\nColumns:")
print(offices.columns.tolist())

Shape:
(10, 7)

First 5 rows:
  office_id          office_name  \
0   OFF-001        Manila Branch   
1   OFF-002   Quezon City Branch   
2   OFF-003        Makati Branch   
3   OFF-101  San Fernando Branch   
4   OFF-102       Angeles Branch   

                                   services_offered contact_phone  \
0  [Business Permit, Civil Registry, Tax Clearance]    8-527-1001   
1               [Business Permit, Zoning Clearance]    8-988-1002   
2                 [Business Permit, Civil Registry]    8-892-1003   
3            [Business Permit, Agricultural Permit]   45-961-1101   
4                   [Civil Registry, Tax Clearance]   45-322-1102   

                       contact_email region_code              region_name  
0       manila.branch@example.gov.ph         NCR  National Capital Region  
1           qc.branch@example.gov.ph         NCR  National Capital Region  
2       makati.branch@example.gov.ph         NCR  National Capital Region  
3  sanfernando.branch@example.gov.

## Lecture 3 — Combining Sources

Merging the flattened JSON with the CSV on a shared key. Inner, left, and
outer joins in pandas, and validating the join.

```python
merged = service_requests.merge(
    offices, how="left", on="office_id",
    indicator=True, validate="many_to_one"
)
```


## Hands-On 3 — Merge and Validate

**Required variable names:** `merged`, `unmatched`, `inner`, `outer`


In [38]:
# Step 1 — Confirm the shared key lines up before merging anything
# Nothing to fill in -- just run this and read the output. If the dtypes
# didn't match (e.g. one side was int and the other was str), THIS is
# where you'd catch it, before it silently fails to match anything.
# (If this errors with "offices is None" or similar, go back and finish
# Hands-On 2 Step 3 first -- this step needs that `offices` DataFrame.)

print("df['office_id'] dtype:     ", df["office_id"].dtype)
print("offices['office_id'] dtype:", offices["office_id"].dtype if offices is not None else "offices not created yet")


df['office_id'] dtype:      object
offices['office_id'] dtype: object


In [39]:
# Step 2 — Merge with indicator
#
# TODO: merge df (service requests) with offices, so that:
#   - every request is kept even if its office doesn't match  (how=?)
#   - the shared key is office_id                              (on=?)
#   - you get a column showing match status                    (indicator=True)
#   - pandas raises an error if an office_id matches more than
#     one office row                                            (validate=?)

merged = df.merge(
    offices,
    on="office_id",
    how="left",
    indicator=True,
    validate="many_to_one",
)


In [40]:
check(
    merged,
    lambda d: d.shape[0] == df.shape[0] and "_merge" in d.columns,
    "row count matches df, and you've got a _merge column -- merge looks right.",
    f"not quite -- a correct left merge here should have exactly {df.shape[0]} rows "
    "(one per service request) and a '_merge' column."
)


Correct - row count matches df, and you've got a _merge column -- merge looks right.


In [41]:
# Step 3 — Count and inspect unmatched rows
#
# TODO: print how many rows fall into each _merge category, then build
#       `unmatched`: just the rows where _merge == "left_only".

# Count how many rows are in each merge category
print(merged["_merge"].value_counts())

# Keep only requests whose office_id did not match an office
unmatched = merged[merged["_merge"] == "left_only"]

# Inspect the unmatched rows
print("\nUnmatched rows:")
print(unmatched)


_merge
both          24
left_only      3
right_only     0
Name: count, dtype: int64

Unmatched rows:
   request_id date_submitted office_id          category  \
24   REQ-0025     2026-08-05   OFF-999   Business Permit   
25   REQ-0026     2026-08-11    OFF-02  Zoning Clearance   
26   REQ-0027     2026-08-14   OFF-999    Civil Registry   

                                          description       status priority  \
24  Old request routed to a since-closed satellite...         Open   Medium   
25  Zoning clearance request with office code ente...  In Progress      Low   
26  Second request affected by the same decommissi...         Open     High   

   office_name services_offered contact_phone contact_email region_code  \
24         NaN              NaN           NaN           NaN         NaN   
25         NaN              NaN           NaN           NaN         NaN   
26         NaN              NaN           NaN           NaN         NaN   

   region_name     _merge  
24         N

In [42]:
check(
    unmatched,
    lambda d: len(d) == 3 and set(d["office_id"]) == {"OFF-999", "OFF-02"},
    "found all 3 unmatched requests, across the 2 bad office_id values.",
    "expected 3 rows, referencing office_id values 'OFF-999' and 'OFF-02'. "
    "Check your filter condition on the _merge column."
)


Correct - found all 3 unmatched requests, across the 2 bad office_id values.


### Step 4 — Interpret the unmatched keys

Look at the `unmatched` rows you just built (print `unmatched[["request_id", "office_id", "description"]]`)
and compare each `office_id` against `offices["office_id"]`.

For each of the two bad codes, decide: is it more likely a **typo** (close
to a real office_id) or a **decommissioned/unmigrated office** (not close
to anything)? Write one sentence for each below.

- `OFF-999` — *(your answer here)*
- `OFF-02` — *(your answer here)*


In [43]:
# Use this cell to inspect the evidence for your answer above.
print(unmatched[["request_id", "office_id", "description"]])
print()
print("Real office_id values:", sorted(offices["office_id"].tolist()))


   request_id office_id                                        description
24   REQ-0025   OFF-999  Old request routed to a since-closed satellite...
25   REQ-0026    OFF-02  Zoning clearance request with office code ente...
26   REQ-0027   OFF-999  Second request affected by the same decommissi...

Real office_id values: ['OFF-001', 'OFF-002', 'OFF-003', 'OFF-101', 'OFF-102', 'OFF-201', 'OFF-202', 'OFF-203', 'OFF-301', 'OFF-302']


In [44]:
# Step 6 — Compare join types
#
# TODO: re-run the same merge as an "inner" join (assign to `inner`) and
#       as an "outer" join (assign to `outer`). Compare their .shape to
#       `merged` (the left join from Step 2) and explain the difference
#       to a partner before moving on.

# Inner join: keeps only rows where office_id exists in BOTH DataFrames
inner = df.merge(
    offices,
    on="office_id",
    how="inner"
)

# Outer join: keeps ALL rows from BOTH DataFrames
outer = df.merge(
    offices,
    on="office_id",
    how="outer"
)

# Compare the shapes
print("Left join (merged): ", merged.shape)
print("Inner join:         ", inner.shape)
print("Outer join:         ", outer.shape)


Left join (merged):  (27, 14)
Inner join:          (24, 13)
Outer join:          (27, 13)


In [45]:
check(
    inner,
    lambda d: d.shape[0] == df.shape[0] - 3,
    f"inner join dropped exactly the 3 unmatched requests ({df.shape[0]} - 3 = {df.shape[0]-3} rows).",
    f"expected {df.shape[0]-3} rows for the inner join (baseline minus the 3 unmatched)."
)
check(
    outer,
    lambda d: d.shape[0] >= merged.shape[0],
    "outer join has at least as many rows as the left join, as expected.",
    "an outer join should never have FEWER rows than the left join -- double-check how='outer'."
)


Correct - inner join dropped exactly the 3 unmatched requests (27 - 3 = 24 rows).
Correct - outer join has at least as many rows as the left join, as expected.


## Bonus — Combining CSV, JSON, *and* SQL

Everything today has come from files. But a lot of real reference data —
category rules, SLAs, department ownership — lives in an actual **database**,
not a flat file. You've already covered reading from SQL in an earlier
session; this section reuses exactly that (`sqlite3` + `pd.read_sql`) and
folds it into the pipeline you just built.

**New piece:** `data/workshop.db`, a SQLite database with one table,
`category_sla`, keyed on `category` — the same `category` column that's
already in `service_requests.csv`.


## Hands-On 4 — Capstone: CSV + JSON + SQL

Same rhythm as before: TODO cell, then check cell.

**Watch for this:** `merged` already carries a `_merge` column from
Hands-On 3. If you reuse `indicator=True` here, pandas will refuse to
create a second column with the same name. Give this merge's indicator
its own name instead (e.g. `indicator="_merge_sla"`).

**Required variable names:** `category_sla`, `full_merge`, `unmatched_categories`


In [46]:
# Step 1 — Connect and read the SQL table
#
# TODO: open a connection to data/workshop.db with sqlite3.connect(), then
#       read the whole category_sla table into a DataFrame with pd.read_sql().
#       (Hint: pd.read_sql("SELECT * FROM category_sla", conn))

import sqlite3

conn = sqlite3.connect("data/workshop.db")

category_sla = pd.read_sql(
    "SELECT * FROM category_sla",
    conn
)

print(category_sla)


              category             department  sla_days  priority_weight
0      Business Permit              Licensing         5                2
1       Civil Registry  Civil Registry Office         3                1
2        Tax Clearance               Treasury         4                2
3     Zoning Clearance               Planning         7                3
4  Agricultural Permit            Agriculture        10                3


In [47]:
check(
    category_sla,
    lambda d: set(d.columns) == {"category", "department", "sla_days", "priority_weight"},
    f"loaded {len(category_sla) if category_sla is not None else 0} rows from category_sla.",
    "expected columns: category, department, sla_days, priority_weight. Check your SQL statement."
)


Correct - loaded 5 rows from category_sla.


In [48]:
# Step 2 — Inspect, the same way you've inspected everything else today
# TODO: run shape, head(), and dtypes on category_sla.
#       How many categories does it cover? Compare that to
#       df["category"].nunique() -- do they match?

print("Shape:")
print(category_sla.shape)

print("\nFirst 5 rows:")
print(category_sla.head())

print("\nData types:")
print(category_sla.dtypes)

print("\nCategories in category_sla:")
print(category_sla["category"].nunique())

print("\nCategories in service requests:")
print(df["category"].nunique())

Shape:
(5, 4)

First 5 rows:
              category             department  sla_days  priority_weight
0      Business Permit              Licensing         5                2
1       Civil Registry  Civil Registry Office         3                1
2        Tax Clearance               Treasury         4                2
3     Zoning Clearance               Planning         7                3
4  Agricultural Permit            Agriculture        10                3

Data types:
category           object
department         object
sla_days            int64
priority_weight     int64
dtype: object

Categories in category_sla:
5

Categories in service requests:
6


In [49]:
# Step 3 — Three-way merge
#
# You already have `merged` (service requests + offices) from Hands-On 3.
# TODO: merge `merged` with `category_sla` on "category", using a LEFT join
#       so you don't lose any request.
#
# One wrinkle: `merged` already has a "_merge" column from Hands-On 3, so
# indicator=True would try to create a column that already exists and
# pandas will raise a ValueError. Give this indicator a NEW name instead,
# e.g. indicator="_merge_sla".

full_merge = merged.merge(
    category_sla,
    on="category",
    how="left",
    indicator="_merge_sla"
)

print(full_merge)


   request_id date_submitted office_id             category  \
0    REQ-0001     2026-07-03   OFF-001      Business Permit   
1    REQ-0002     2026-07-08   OFF-002       Civil Registry   
2    REQ-0003     2026-07-12   OFF-003        Tax Clearance   
3    REQ-0004     2026-07-15   OFF-101     Zoning Clearance   
4    REQ-0005     2026-07-19   OFF-102  Agricultural Permit   
5    REQ-0006     2026-07-22   OFF-201            Complaint   
6    REQ-0007     2026-07-27   OFF-202      Business Permit   
7    REQ-0008     2026-08-03   OFF-203       Civil Registry   
8    REQ-0009     2026-08-08   OFF-301        Tax Clearance   
9    REQ-0010     2026-08-12   OFF-302     Zoning Clearance   
10   REQ-0011     2026-08-15   OFF-001  Agricultural Permit   
11   REQ-0012     2026-08-19   OFF-002            Complaint   
12   REQ-0013     2026-08-22   OFF-003      Business Permit   
13   REQ-0014     2026-08-27   OFF-101       Civil Registry   
14   REQ-0015     2026-07-03   OFF-102        Tax Clear

In [50]:
check(
    full_merge,
    lambda d: d.shape[0] == merged.shape[0] and "sla_days" in d.columns,
    "row count still matches `merged`, and sla_days is now a column -- three sources combined.",
    f"expected {merged.shape[0] if 'merged' in dir() else '?'} rows and an sla_days column. "
    "Check on= and how= again."
)


Correct - row count still matches `merged`, and sla_days is now a column -- three sources combined.


In [51]:
# Step 4 — Find the unmatched categories
#
# TODO: filter full_merge to the rows where the SQL side didn't match.
#       Use the indicator column name YOU chose in Step 3 (not "_merge" --
#       that one already belongs to the office merge).
#       Assign to unmatched_categories.

unmatched_categories = full_merge[
    full_merge["_merge_sla"] == "left_only"
]

print(unmatched_categories)


   request_id date_submitted office_id   category  \
5    REQ-0006     2026-07-22   OFF-201  Complaint   
11   REQ-0012     2026-08-19   OFF-002  Complaint   
17   REQ-0018     2026-07-15   OFF-203  Complaint   
23   REQ-0024     2026-08-12   OFF-101  Complaint   

                                   description       status priority  \
5   Request for certified true copy of records  In Progress     High   
11            Application for new registration       Closed     High   
17      Complaint regarding delayed processing  In Progress     High   
23   Cafe owner requesting inspection schedule       Closed     High   

            office_name                                   services_offered  \
5        Calamba Branch  [Business Permit, Zoning Clearance, Tax Cleara...   
11   Quezon City Branch                [Business Permit, Zoning Clearance]   
17      Antipolo Branch             [Business Permit, Agricultural Permit]   
23  San Fernando Branch             [Business Permit, Agricul

In [52]:
check(
    unmatched_categories,
    lambda d: set(d["category"].unique()) == {"Complaint"},
    "found it -- every unmatched row is a Complaint, which has no SLA row on purpose.",
    "expected every unmatched row's category to be 'Complaint' -- that's the one category "
    "missing from category_sla. Check your filter."
)


Correct - found it -- every unmatched row is a Complaint, which has no SLA row on purpose.


### Step 5 — Why would a category legitimately have no SLA?

You've now seen two kinds of unmatched keys today: a **typo** (`OFF-02`),
a **decommissioned record** (`OFF-999`), and now a **category that was
never meant to be in this table** (`Complaint`). In one or two sentences,
how would you explain to a teammate why "no match" doesn't always mean
"something is broken"?

*(your answer here)*


In [53]:
# Nothing to fill in -- just close the loop and look at the final shape.
# (If full_merge is None, go back and finish Step 3 above first.)
if full_merge is not None:
    print("Final combined dataset:", full_merge.shape)
    print(full_merge.columns.tolist())
else:
    print("full_merge not created yet -- finish Step 3 first.")

conn.close()

Final combined dataset: (27, 18)
['request_id', 'date_submitted', 'office_id', 'category', 'description', 'status', 'priority', 'office_name', 'services_offered', 'contact_phone', 'contact_email', 'region_code', 'region_name', '_merge', 'department', 'sla_days', 'priority_weight', '_merge_sla']


## Recap & Formative Check

- ✅ Import CSV and JSON into pandas
- ✅ Flatten nested JSON into tabular form
- ✅ Merge two sources on a key and validate the result
- ✅ Bonus: bring a SQL source into the same pipeline as CSV and JSON

**Quick check:**
1. Your CSV import puts every value into a single column. Likely cause, and the fix?
2. What's the difference between `pd.read_json()` and `pd.json_normalize()`?
3. In `pd.json_normalize()`, what does `record_path` control?
4. A left merge produces *more* rows than your left DataFrame. What does that tell you?
5. Name two things a `left_only` row could mean in the service-request/office scenario.
6. In the capstone merge, every unmatched row was "Complaint" instead of a random mix
   of categories. Why does that pattern matter more than the raw count of unmatched rows?
